In [1]:
# ============================================================
# FULL STEADY-STATE REACTOR + PARAMETRIC + MULTI-KINETICS
# ============================================================

from fipy import *
import numpy as np
import matplotlib.pyplot as plt
from fipy.tools import numerix
import pandas as pd
from fipy.terms import ImplicitSourceTerm


def safe(x):
    return numerix.maximum(x, 1e-8)

def safe_exp(x):
    return numerix.exp(numerix.clip(x, -100, 100))

def safe_div(numer, denom, eps=1e-12):
    return numer / numerix.maximum(denom, eps)

def radial_avg(Cvar, Nr, Nz):
    """Return radial average along reactor length (axis=0 is radial)."""
    return np.mean(Cvar.reshape(Nr, Nz), axis=0)

# -----------------------------
# Global constants
# -----------------------------
R = 8.314
rho_b = 2450
eps = 0.4
Dr, Dz = 1e-5, 1e-5
lambda_e = 0.8
dH = [-165e3, 41e3, -206e3]
Rr, Lz = 0.1, 1
Nr, Nz = 20, 60
Tin = 600

species = ["H2", "CO2", "CH4", "H2O", "CO"]

nu = {
    "H2":  [-4, -1, -3],
    "CO2": [-1, -1,  0],
    "CH4": [ 1,  0,  1],
    "H2O": [ 2,  1,  1],
    "CO":  [ 0,  1, -1]
}


# ============================================================
# KINETIC MODELS
# ============================================================

def champom_kinetics(p, T, k):
    T = numerix.clip(T, 300, 2000)

    kco2  = k["k0co2"]  * safe_exp(-k["eaco2"] / (R*T))
    krwgs = k["k0rwgs"] * safe_exp(-k["earwgs"] / (R*T))
    kco   = k["k0co"]   * safe_exp(-k["eaco"] / (R*T))

    Keq = 137* T**(-3.998) * safe_exp(158700/(R*T))

    Kco  = k["kads0co"]  * safe_exp(k["deltahco"]  / (R*T))
    Kh2  = k["kads0h2"]  * safe_exp(k["deltahh2"]  / (R*T))
    Kh2o = k["kads0h2o"] * safe_exp(k["deltahh2o"] / (R*T))
    Kco2 = k["kads0co2"] * safe_exp(k["deltahco2"] / (R*T))

    denom = 1 + Kh2 * p["H2"] + Kco2 * p["CO2"] + Kh2o * p["H2O"] + Kco * p["CO"]

    driving = safe_div(1 - (p["CH4"] * p["H2O"]**2) / (p["CO2"] * p["H2"]**4 * Keq), 1)
    driving = numerix.clip(driving, 0, 1)

    r1 = kco2 * Kh2 * Kco2 * p["H2"] * p["CO2"] * driving / numerix.maximum(denom**2, 1e-12)
    r2 = krwgs * p["CO2"] / numerix.maximum(denom, 1e-12)
    r3   = kco * p["CO"] * p["H2"] / numerix.maximum(denom**2, 1e-12)
    
    return [r1, r2, r3]

def koz_kinetics(p, T, k):
    T = numerix.clip(T, 300, 2000)

    kf = k["k0"] * safe_exp(-k["Ea"] / (R*T))
    Keq = 137 * T**(-3.998) * safe_exp(158700 / (R*T))

    K_OH  = numerix.exp(k["Aoh"]  + k["Boh"] / T)
    K_H2  = numerix.exp(k["Ah2"]  + k["Bh2"] / T)
    K_mix = numerix.exp(k["Amix"] + k["Bmix"] / T)

    driving = safe_div(1 - (p["CH4"] * p["H2O"]**2)/(p["CO2"] * p["H2"]**4 * Keq),1)
    driving = numerix.clip(driving,0,1)

    num = kf * p["H2"]**0.5 * p["CO2"]**0.5 * driving
    den = (1 + K_OH*(p["H2O"] / p["H2"]**0.5) + K_H2*p["H2"]**0.5 + K_mix*p["CO2"]**0.5)**2

    return safe_div(num, den), 0.0, 0.0

def run_reactor(H2_CO2, Twall, u0, Pbar, kinetics, kin_params):

    # ---- geometry
    Rr, Lz = 0.1, 1.0
    Nr, Nz = 200, 600
    mesh = CylindricalGrid2D(dr=Rr/Nr, dz=Lz/Nz, nr=Nr, nz=Nz)

    # ---- constants
    Tin = 600.0
    eps = 0.4
    Dr, Dz = 2e-5, 2e-5
    rho_b = 1000.0
    rhoCp = 1.5e6
    lambda_e = 1.0

    # ---- inlet composition
    yCO2 = 1 / (1 + H2_CO2)
    yH2  = H2_CO2 * yCO2

    Cin = {
        "CO2": Pbar * 1e5 * yCO2 / (R * Tin),
        "H2":  Pbar * 1e5 * yH2  / (R * Tin),
        "CH4": 1e-12,
        "CO":  1e-12,
        "H2O": 1e-12
    }

    # ---- variables
    C = {sp: CellVariable(mesh=mesh, value=Cin[sp], hasOld=True)
         for sp in species}
    T = CellVariable(mesh=mesh, value=Tin, hasOld=True)

    # ---- boundary conditions
    for sp in species:
        C[sp].constrain(Cin[sp], mesh.facesBottom)
        C[sp].faceGrad.constrain(0, mesh.facesTop)

    T.constrain(Tin, mesh.facesBottom)
    T.constrain(Twall, mesh.facesTop)

    # ---- partial pressures
    p = {sp: safe(C[sp] * R * T / 1e5) for sp in species}

    # ---- reaction rates
    rates = kinetics(p, T, kin_params)

    # ==========================================================
    # SPECIES EQUATIONS
    # ==========================================================
    eqs = []
    for sp in species:

        Rsp = sum(nu[sp][i] * rates[i] for i in range(3))

        eq = (
            TransientTerm(var=C[sp])
            + ConvectionTerm(coeff=(0, u0), var=C[sp])
            ==
            DiffusionTerm(coeff=(eps * Dr, eps * Dz), var=C[sp])
            + ImplicitSourceTerm(
                coeff=rho_b * Rsp / (C[sp] + 1e-12),
                var=C[sp]
            )
        )
        eqs.append(eq)

    # ==========================================================
    # ENERGY EQUATION
    # ==========================================================
    Qrxn = sum(-dH[i] * rates[i] for i in range(3))

    energy = (
        TransientTerm(coeff=rhoCp, var=T)
        + ConvectionTerm(coeff=(0, rhoCp * u0), var=T)
        ==
        DiffusionTerm(coeff=lambda_e, var=T)
        + ImplicitSourceTerm(
            coeff=rho_b * Qrxn / (T + 1e-6),
            var=T
        )
    )

    # ==========================================================
    # TIME INTEGRATION
    # ==========================================================
    dt = 1e-6
    for step in range(400):

        for sp in species:
            C[sp].updateOld()
        T.updateOld()

        for eq in eqs:
            eq.solve(dt=dt)

        energy.solve(dt=dt)

        Tmax = T.value.max()
        if step % 20 == 0:
            print(f"step {step:4d} | Tmax = {Tmax:8.1f} K")

        # ---- adaptive timestep
        if Tmax > 2000:
            dt *= 0.5
        elif Tmax < 1200:
            dt *= 1.05

        dt = max(1e-8, min(dt, 1e-3))

    # ---- postprocessing
    cl = lambda v: v.value.reshape(Nr, Nz)[0, :]
    CO2_in, CO2_out = Cin["CO2"], cl(C["CO2"])[-1]
    CH4_out = cl(C["CH4"])[-1]

    X = (CO2_in - CO2_out) / CO2_in
    S = CH4_out / (CO2_in - CO2_out + 1e-12)

    return X, S, T.value.reshape(Nr, Nz)

# ============================================================
# PARAMETERS
# ============================================================

# ---- Champom kinetics ----
kin_champom = dict(
    k0co2=1900000, eaco2=110000,
    k0rwgs=29666.66667, earwgs=97100,
    k0co=3716666.667, eaco=97300,
    kads0co=0.00239, deltahco=40600,
    kads0h2=0.000052, deltahh2=52000,
    kads0h2o=0.609, deltahh2o=14500,
    kads0co2=1.07, deltahco2=9720
)

kin_koz = dict(
    k0=3.46e-5,
    Ea=77500,
    Aoh=0.5, Boh=-2694.25,
    Ah2=0.44, Bh2=745.73,
    Amix=0.88, Bmix=1202.79
)

ratios = [3.5, 4]
Twalls = [600, 650]
vels = [0.1, 0.5]
pressures = [7, 10]

# ============================================================
# PARAMETRIC SWEEP
# ============================================================

results = []

case = 0
total = len(ratios) * len(Twalls) * len(vels) * len(pressures)

for r in ratios:
    for Tw in Twalls:
        for u in vels:
            for P in pressures:
                case += 1
                print(f"▶ Case {case} / {total} | H2/CO2={r}, T={Tw}, u={u}, P={P}")

                Xc, Sc, Tc = run_reactor(r, Tw, u, P,
                                     champom_kinetics, kin_champom)
                Xk, Sk, Tk = run_reactor(r, Tw, u, P,
                                     koz_kinetics, kin_koz)
                results.append([r, Tw, u, P, Xc, Sc, Xk, Sk])

results = pd.DataFrame(
    results,
    columns=["H2_CO2", "Twall", "u0", "P",
             "X_champom", "S_champom", "X_Koz", "S_Koz"]
)

results

# ============================================================
# PLOTS
# ============================================================

# Convert conversion and selectivity to percent
results["X_ch_pct"] = results["X_champom"] * 100
results["X_koz_pct"] = results["X_Koz"] * 100
results["S_ch_pct"] = results["S_champom"] * 100
results["S_koz_pct"] = results["S_Koz"] * 100

# --- CO2 Conversion vs H2/CO2 ratio ---
plt.figure()
for Tw in Twalls:
    sub = results[results.Twall == Tw]
    plt.plot(sub.H2_CO2, sub.X_ch_pct, "o-", label=f"Champom, Tw={Tw}K")
    plt.plot(sub.H2_CO2, sub.X_koz_pct, "s--", label=f"Koz, Tw={Tw}K")
plt.xlabel("H2/CO2 ratio")
plt.ylabel("CO₂ conversion [%]")
plt.title("CO₂ Conversion vs H2/CO2 ratio")
plt.legend()
plt.grid(True)
plt.show()

# --- CO2 Conversion vs Wall Temperature ---
for u in results.u0.unique():
    sub = results[results.u0 == u]
    plt.figure()
    plt.plot(sub.Twall, sub.X_ch_pct, 'o-', label="Champom")
    plt.plot(sub.Twall, sub.X_koz_pct, 's--', label="Koz")
    plt.xlabel("Wall temperature [K]")
    plt.ylabel("CO₂ conversion [%]")
    plt.title(f"Effect of Temperature (u₀ = {u} m/s)")
    plt.legend()
    plt.grid(True)
    plt.show()

# --- CO2 Conversion vs Pressure ---
for Tw in results.Twall.unique():
    sub = results[results.Twall == Tw]
    plt.figure()
    plt.plot(sub.P, sub.X_ch_pct, 'o-', label="Champom")
    plt.plot(sub.P, sub.X_koz_pct, 's--', label="Koz")
    plt.xlabel("Pressure [bar]")
    plt.ylabel("CO₂ conversion [%]")
    plt.title(f"Effect of Pressure (Twall = {Tw} K)")
    plt.legend()
    plt.grid(True)
    plt.show()

# --- CO2 Conversion vs Velocity ---
for P in results.P.unique():
    sub = results[results.P == P]
    plt.figure()
    plt.plot(sub.u0, sub.X_ch_pct, 'o-', label="Champom")
    plt.plot(sub.u0, sub.X_koz_pct, 's--', label="Koz")
    plt.xlabel("Superficial velocity u₀ [m/s]")
    plt.ylabel("CO₂ conversion [%]")
    plt.title(f"Effect of Velocity (P = {P} bar)")
    plt.legend()
    plt.grid(True)
    plt.show()

# --- Selectivity vs Conversion ---
plt.figure()
plt.scatter(results.X_ch_pct, results.S_ch_pct, c=results.Twall,
            cmap="inferno", label="Champom")
plt.colorbar(label="Twall [K]")
plt.xlabel("CO₂ conversion [%]")
plt.ylabel("CO selectivity [%]")
plt.title("Selectivity vs Conversion (Champom)")
plt.grid(True)
plt.show()

▶ Case 1 / 16 | H2/CO2=3.5, T=600, u=0.1, P=7


C:\Users\pingu\anaconda3\Lib\site-packages\fipy\variables\variable.py:1130: RuntimeWarning: invalid value encountered in divide
  return self._BinaryOperatorVariable(lambda a, b: a / b, other)


step    0 | Tmax =    600.0 K


KeyboardInterrupt: 